# Programming Assignment: Feature Creation & Engineering

Welcome to the Feature Creation & Engineering Programming Assignment!

Raw data seldom tells the full story. Feature engineering — the art of creating new, meaningful variables from existing ones — is often the single most impactful step in building high-performance models. In this assignment you will build features that no automatic algorithm could infer: temporal patterns, text signals, and domain-driven aggregates.

**What You Will Do in this Assignment:**

* **RFM Features** - Engineer Recency, Frequency, and Monetary features from transaction timestamps and amounts
* **Mathematical Transforms** - Create ratio features, log-transformed values, and pairwise interaction terms
* **Datetime Features** - Extract hour-of-day, day-of-week, month, and is-weekend flags from timestamp columns
* **Text Features** - Parse product name strings into keyword indicators, word counts, and presence flags
* **Polynomial Features (sklearn)** - Use `PolynomialFeatures` to systematically generate interaction and power terms
* **Polynomial Features from Scratch** - Implement polynomial feature generation using `itertools.combinations_with_replacement`

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

- [Exercise 1 – Domain Features (RFM)](#exercise-1)
- [Exercise 2 – Mathematical Features](#exercise-2)
- [Exercise 3 – DateTime Features](#exercise-3)
- [Exercise 4 – Text Features](#exercise-4)
- [Exercise 5 – Polynomial Features (sklearn)](#exercise-5)
- [Exercise 6 – Polynomial Features from Scratch](#exercise-6)

## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
from itertools import combinations_with_replacement
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Dataset

We will use a synthetic customer transaction dataset with 500 customers. Each row represents one customer with:

| Column | Description |
|:-------|:------------|
| `customer_id` | Unique customer identifier |
| `last_purchase_date` | Date of most recent purchase |
| `purchase_count` | Total number of purchases made |
| `total_spent` | Total amount spent (in currency units) |
| `n_products` | Number of distinct product types purchased |
| `product_name` | Name of the most recently purchased product |
| `signup_date` | Date customer first registered |

All datetime-based features will use `REFERENCE_DATE = pd.Timestamp('2024-01-01')` as the fixed reference point.

In [ ]:
np.random.seed(42)
n_customers = 500
REFERENCE_DATE = pd.Timestamp('2024-01-01')

last_purchase_dates = REFERENCE_DATE - pd.to_timedelta(
    np.random.randint(1, 365, n_customers), unit='D')
signup_dates = REFERENCE_DATE - pd.to_timedelta(
    np.random.randint(365, 1825, n_customers), unit='D')

df = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'last_purchase_date': last_purchase_dates,
    'purchase_count': np.random.randint(1, 50, n_customers),
    'total_spent': np.random.uniform(10, 5000, n_customers).round(2),
    'n_products': np.random.randint(1, 20, n_customers),
    'product_name': np.random.choice([
        'laptop computer', 'wireless headphones', 'coffee maker deluxe',
        'running shoes premium', 'smartphone case', 'gaming keyboard rgb'
    ], n_customers),
    'signup_date': signup_dates,
})
X_num = df[['purchase_count', 'total_spent', 'n_products']].values.astype(float)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())

Dataset shape: (500, 7)

First 3 rows:
   customer_id last_purchase_date  purchase_count  total_spent  n_products        product_name signup_date
0            1        2023-09-20              49      2844.05          15     smartphone case  2021-08-23
1            2        2023-01-17              12      4578.29           4  running shoes premium  2020-07-20
2            3        2023-04-05              47       179.39          11   gaming keyboard rgb  2022-07-26

<a id='exercise-1'></a>
## Exercise 1 – Domain Features (RFM)

### Background

The **RFM** framework captures customer behavior in three dimensions:
- **Recency** — how recently did they buy? (smaller = more engaged)
- **Frequency** — how often do they buy? (higher = more loyal)
- **Monetary** — how much do they spend? (higher = more valuable)

These three metrics are the foundation of customer segmentation in retail, e-commerce, and subscription businesses.

### Task

Implement `create_domain_features(df, reference_date)` that adds **4 new columns** to a copy of `df`:

| New column | Formula |
|:-----------|:--------|
| `recency_days` | `(reference_date − last_purchase_date).days` |
| `avg_order_value` | `total_spent / purchase_count` |
| `customer_lifetime_days` | `(reference_date − signup_date).days` |
| `purchase_frequency` | `purchase_count / customer_lifetime_days` |

**Hints:**
- Use `.dt.days` on a timedelta Series to get integer days.
- Make a `result = df.copy()` first so you don't modify the original DataFrame.
- `reference_date` has a default value of `pd.Timestamp('2024-01-01')`. Use it as-is; do not recompute it from the data.

In [ ]:
def create_domain_features(df, reference_date=pd.Timestamp('2024-01-01')):
    """
    Create domain-driven RFM features from customer transaction data.

    Args:
        df (pd.DataFrame): Customer DataFrame with columns:
            last_purchase_date, purchase_count, total_spent, signup_date
        reference_date (pd.Timestamp): Fixed date to compute recency from.

    Returns:
        pd.DataFrame: Copy of df with 4 new columns:
            recency_days          – integer, days since last purchase
            avg_order_value       – float, total_spent / purchase_count
            customer_lifetime_days – integer, days since signup
            purchase_frequency    – float, purchase_count / customer_lifetime_days
    """
    # YOUR CODE HERE
    return None

In [ ]:
df_domain = create_domain_features(df)
print(f"Shape before: {df.shape}  |  Shape after: {df_domain.shape}")
print(f"New columns: {[c for c in df_domain.columns if c not in df.columns]}")
print(f"\nFirst 3 rows of new features:")
print(df_domain[['recency_days', 'avg_order_value', 'customer_lifetime_days', 'purchase_frequency']].head(3).to_string())

Shape before: (500, 7)  |  Shape after: (500, 11)
New columns: ['recency_days', 'avg_order_value', 'customer_lifetime_days', 'purchase_frequency']

First 3 rows of new features:
   recency_days  avg_order_value  customer_lifetime_days  purchase_frequency
0           103        58.041837                     861            0.056911
1           349       381.524167                    1260            0.009524
2           271         3.816809                     524            0.089695

In [ ]:
unittests.exercise_1(create_domain_features)

<a id='exercise-2'></a>
## Exercise 2 – Mathematical Features

### Background

Mathematical transformations let linear models learn non-linear relationships and normalize away size effects:
- **Ratios** normalize one quantity by another related one (e.g. spend per product)
- **Interaction terms** (A × B) let the model respond to the combined effect of two features
- **Log transforms** compress right-skewed distributions; `np.log1p(x)` is safe when x = 0
- **Squared terms** give the model the ability to fit curves

### Task

Implement `create_mathematical_features(df)` that adds **4 new columns**:

| New column | Formula |
|:-----------|:--------|
| `spend_per_product` | `total_spent / n_products` |
| `purchase_spend_interaction` | `purchase_count × total_spent` |
| `log_total_spent` | `np.log1p(total_spent)` |
| `total_spent_squared` | `total_spent ** 2` |

**Hints:**
- Make a copy with `result = df.copy()`.
- Use `np.log1p`, not `np.log` — it handles zero values safely.

In [ ]:
def create_mathematical_features(df):
    """
    Create mathematical transformation features.

    Args:
        df (pd.DataFrame): Customer DataFrame with columns:
            total_spent, n_products, purchase_count

    Returns:
        pd.DataFrame: Copy of df with 4 new columns:
            spend_per_product          – float, total_spent / n_products
            purchase_spend_interaction – float, purchase_count * total_spent
            log_total_spent            – float, np.log1p(total_spent)
            total_spent_squared        – float, total_spent ** 2
    """
    # YOUR CODE HERE
    return None

In [ ]:
df_math = create_mathematical_features(df)
print(f"New columns: {[c for c in df_math.columns if c not in df.columns]}")
print(f"\nFirst 3 rows of new features:")
print(df_math[['spend_per_product', 'purchase_spend_interaction', 'log_total_spent', 'total_spent_squared']].head(3).to_string())

New columns: ['spend_per_product', 'purchase_spend_interaction', 'log_total_spent', 'total_spent_squared']

First 3 rows of new features:
   spend_per_product  purchase_spend_interaction  log_total_spent  total_spent_squared
0         189.603333                   139358.45         7.953336         8.088620e+06
1        1144.572500                    54939.48         8.429299         2.096074e+07
2          16.308182                     8431.33         5.195121         3.218077e+04

In [ ]:
unittests.exercise_2(create_mathematical_features)

<a id='exercise-3'></a>
## Exercise 3 – DateTime Features

### Background

A raw datetime column is just an integer — a linear model cannot directly use "September 20, 2023" to learn that autumn purchases differ from spring purchases. Extracting temporal components breaks the datetime into independent signals:

- **Day of week** captures within-week patterns (Monday shoppers vs. weekend shoppers)
- **Month / Quarter** captures seasonality
- **Is weekend** collapses the day-of-week signal into a binary flag
- **Year** captures long-term trends

### Task

Implement `create_datetime_features(df)` that adds **5 new columns**:

| New column | Source | Method |
|:-----------|:-------|:-------|
| `purchase_day_of_week` | `last_purchase_date` | `.dt.dayofweek` (0=Mon, 6=Sun) |
| `purchase_month` | `last_purchase_date` | `.dt.month` |
| `purchase_quarter` | `last_purchase_date` | `.dt.quarter` |
| `is_weekend_purchase` | `last_purchase_date` | 1 if `dayofweek >= 5` else 0 |
| `signup_year` | `signup_date` | `.dt.year` |

**Hints:**
- Use `.astype(int)` to convert the boolean weekend condition to 0/1.
- All five features come from the two datetime columns — no arithmetic needed.

In [ ]:
def create_datetime_features(df):
    """
    Extract temporal features from datetime columns.

    Args:
        df (pd.DataFrame): Customer DataFrame with columns:
            last_purchase_date, signup_date

    Returns:
        pd.DataFrame: Copy of df with 5 new columns:
            purchase_day_of_week – int, 0=Monday … 6=Sunday
            purchase_month       – int, 1–12
            purchase_quarter     – int, 1–4
            is_weekend_purchase  – int, 1 if Sat/Sun else 0
            signup_year          – int
    """
    # YOUR CODE HERE
    return None

In [ ]:
df_dt = create_datetime_features(df)
print(f"New columns: {[c for c in df_dt.columns if c not in df.columns]}")
print(f"\nFirst 3 rows of new features:")
print(df_dt[['purchase_day_of_week', 'purchase_month', 'purchase_quarter', 'is_weekend_purchase', 'signup_year']].head(3).to_string())
print(f"\nWeekend purchases: {df_dt['is_weekend_purchase'].sum()} out of {len(df_dt)}")

New columns: ['purchase_day_of_week', 'purchase_month', 'purchase_quarter', 'is_weekend_purchase', 'signup_year']

First 3 rows of new features:
   purchase_day_of_week  purchase_month  purchase_quarter  is_weekend_purchase  signup_year
0                     2               9                 3                    0         2021
1                     1               1                 1                    0         2020
2                     2               4                 2                    0         2022

Weekend purchases: 154 out of 500

In [ ]:
unittests.exercise_3(create_datetime_features)

<a id='exercise-4'></a>
## Exercise 4 – Text Features

### Background

Raw text cannot be fed directly to most ML models. The simplest approach is to extract numeric proxy features that capture meaningful aspects of the text:

- **Length** and **word count** correlate with product complexity and description quality
- **Keyword flags** capture product category signals (premium, wireless) as binary indicators

For richer representations you would use TF-IDF or word embeddings, but for many applications these simple features are surprisingly effective.

### Task

Implement `create_text_features(df)` that adds **4 new columns** from `product_name`:

| New column | How to compute |
|:-----------|:--------------|
| `name_length` | `str.len()` |
| `word_count` | `str.split().str.len()` |
| `has_premium` | 1 if name contains "premium" (case-insensitive) else 0 |
| `has_wireless` | 1 if name contains "wireless" (case-insensitive) else 0 |

**Hints:**
- Use `str.contains('premium', case=False)` for case-insensitive search.
- Convert the boolean result to int with `.astype(int)`.

In [ ]:
def create_text_features(df):
    """
    Extract numerical features from a text column.

    Args:
        df (pd.DataFrame): Customer DataFrame with column:
            product_name

    Returns:
        pd.DataFrame: Copy of df with 4 new columns:
            name_length  – int, number of characters in product_name
            word_count   – int, number of words in product_name
            has_premium  – int binary, 1 if 'premium' in name (case-insensitive) else 0
            has_wireless – int binary, 1 if 'wireless' in name (case-insensitive) else 0
    """
    # YOUR CODE HERE
    return None

In [ ]:
df_text = create_text_features(df)
print(f"New columns: {[c for c in df_text.columns if c not in df.columns]}")
print(f"\nFirst 5 rows:")
print(df_text[['product_name', 'name_length', 'word_count', 'has_premium', 'has_wireless']].head(5).to_string())
print(f"\nProducts with 'premium': {df_text['has_premium'].sum()}")
print(f"Products with 'wireless': {df_text['has_wireless'].sum()}")

New columns: ['name_length', 'word_count', 'has_premium', 'has_wireless']

First 5 rows:
            product_name  name_length  word_count  has_premium  has_wireless
0        smartphone case           15           2            0             0
1  running shoes premium           21           3            1             0
2    gaming keyboard rgb           19           3            0             0
3        laptop computer           15           2            0             0
4        laptop computer           15           2            0             0

Products with 'premium': 100
Products with 'wireless': 67

In [ ]:
unittests.exercise_4(create_text_features)

### Bonus: TF-IDF demo (not graded)

Here is how you would create richer text features using TF-IDF. This cell is for exploration — it is not tested.

In [ ]:
vectorizer = TfidfVectorizer(max_features=10)
tfidf_matrix = vectorizer.fit_transform(df['product_name'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)
print("TF-IDF feature matrix shape:", tfidf_df.shape)
print("\nFeature names:", list(tfidf_df.columns))
print("\nFirst 3 rows:")
print(tfidf_df.head(3).round(4).to_string())

<a id='exercise-5'></a>
## Exercise 5 – Polynomial Features (scikit-learn)

### Background

`PolynomialFeatures` from scikit-learn systematically generates all polynomial and interaction terms up to a given degree. For 3 input features [x₁, x₂, x₃] at degree 2:

$$[x_1, x_2, x_3] \rightarrow [x_1, x_2, x_3, x_1^2, x_1 x_2, x_1 x_3, x_2^2, x_2 x_3, x_3^2]$$

This gives a model the ability to learn non-linear relationships and feature interactions automatically.

**Warning:** Feature count grows combinatorially. Always apply regularization or feature selection after using `PolynomialFeatures`.

### Task

Implement `create_polynomial_features(X, degree=2)` using `sklearn.preprocessing.PolynomialFeatures`:
- Set `include_bias=False` (we don't want a constant term)
- Fit and transform `X` in one step with `.fit_transform(X)`

**Hints:**
- The function takes a NumPy array `X`, not a DataFrame.
- For 3 input features and degree=2, the output has 9 columns.
- For 3 input features and degree=3, the output has 19 columns.

In [ ]:
def create_polynomial_features(X, degree=2):
    """
    Generate polynomial and interaction features using scikit-learn.

    Args:
        X (np.ndarray): Input feature matrix, shape (n_samples, n_features)
        degree (int): Maximum polynomial degree.

    Returns:
        np.ndarray: Transformed feature matrix.
            For 3 input features and degree=2, output shape is (n_samples, 9).
    """
    # YOUR CODE HERE
    return None

In [ ]:
X_poly = create_polynomial_features(X_num, degree=2)
print(f"Input shape:          {X_num.shape}")
print(f"Output shape (deg=2): {X_poly.shape}")
print(f"Output shape (deg=3): {create_polynomial_features(X_num, degree=3).shape}")
print(f"\nFirst row — original: {X_num[0]}")
print(f"First row — poly:     {X_poly[0]}")

Input shape:          (500, 3)
Output shape (deg=2): (500, 9)
Output shape (deg=3): (500, 19)

First row — original: [  49.   2844.05   15.  ]
First row — poly:     [4.90000000e+01 2.84405000e+03 1.50000000e+01 2.40100000e+03
 1.39358450e+05 7.35000000e+02 8.08862002e+06 4.26607500e+04
 2.25000000e+02]

In [ ]:
unittests.exercise_5(create_polynomial_features)

<a id='exercise-6'></a>
## Exercise 6 – Polynomial Features from Scratch

### Background

Now implement the same transformation without scikit-learn.

The key insight is that each degree-d polynomial term is the product of d column indices from the input. The function `itertools.combinations_with_replacement(range(n_features), d)` generates all such index tuples in sorted order — exactly matching what sklearn produces.

**Algorithm:**
1. Start with `feature_cols = [X]`  (the original features)
2. For each degree d from 2 to `degree`:
   - For each combination of d column indices (with repetition allowed):
     - Compute the new column as `np.prod([X[:, i] for i in combo], axis=0)`
     - Reshape to `(-1, 1)` and append to `feature_cols`
3. Return `np.hstack(feature_cols)`

### Task

Implement `polynomial_features_from_scratch(X, degree=2)`.

**Hints:**
- `combinations_with_replacement(range(3), 2)` yields: `(0,0), (0,1), (0,2), (1,1), (1,2), (2,2)` — exactly the 6 degree-2 terms.
- `np.prod([X[:, 0], X[:, 1]], axis=0)` computes the element-wise product of two columns.
- Your output must match `sklearn.preprocessing.PolynomialFeatures(degree=..., include_bias=False).fit_transform(X)` exactly.

In [ ]:
def polynomial_features_from_scratch(X, degree=2):
    """
    Generate polynomial and interaction features WITHOUT using sklearn.

    Use itertools.combinations_with_replacement to enumerate all degree-d
    feature combinations and np.prod to compute each new term.

    Args:
        X (np.ndarray): Input feature matrix, shape (n_samples, n_features)
        degree (int): Maximum polynomial degree.

    Returns:
        np.ndarray: Transformed feature matrix matching sklearn's output exactly.
    """
    # YOUR CODE HERE
    return None

In [ ]:
X_scratch = polynomial_features_from_scratch(X_num, degree=2)
X_sklearn = create_polynomial_features(X_num, degree=2)

print(f"Shape (from scratch):  {X_scratch.shape}")
print(f"Shape (sklearn):       {X_sklearn.shape}")
print(f"Matches sklearn:       {np.allclose(X_scratch, X_sklearn)}")
print(f"\nFirst row — from scratch: {X_scratch[0]}")

Shape (from scratch):  (500, 9)
Shape (sklearn):       (500, 9)
Matches sklearn:       True

First row — from scratch: [4.90000000e+01 2.84405000e+03 1.50000000e+01 2.40100000e+03
 1.39358450e+05 7.35000000e+02 8.08862002e+06 4.26607500e+04
 2.25000000e+02]

In [ ]:
unittests.exercise_6(polynomial_features_from_scratch)

## Summary

You have engineered six sets of features from raw customer transaction data:

| Exercise | Features created | Technique |
|:---------|:----------------|:----------|
| 1 | `recency_days`, `avg_order_value`, `customer_lifetime_days`, `purchase_frequency` | Domain knowledge (RFM) |
| 2 | `spend_per_product`, `purchase_spend_interaction`, `log_total_spent`, `total_spent_squared` | Mathematical transforms |
| 3 | `purchase_day_of_week`, `purchase_month`, `purchase_quarter`, `is_weekend_purchase`, `signup_year` | Datetime extraction |
| 4 | `name_length`, `word_count`, `has_premium`, `has_wireless` | Text features |
| 5 | 9-column polynomial matrix | `sklearn.PolynomialFeatures` |
| 6 | 9-column polynomial matrix | `combinations_with_replacement` |

**Key takeaways:**
- Good features beat complex models almost every time.
- Always use a fixed reference date for time-based features.
- Log transforms should be applied before scaling.
- Polynomial features explode in size — use regularization.